# Schema Fetcher — Single MCP Tool

Calls one MCP server tool that handles the full flow:
1. Fetch stored procedure DDL
2. Extract ETL content
3. Return structured success/error result

Tool used: `fetch_stored_procedure_etl`

In [1]:

# -- Imports ---------------------------------------------------------------
import sys, json, pathlib, importlib

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # expose mcp_server

import mcp_server
importlib.reload(mcp_server)

# Single MCP tool call for full schema fetch flow
fetch_stored_procedure_etl = mcp_server.fetch_stored_procedure_etl

print("Imports OK")


Imports OK


In [3]:

# -- Run the script ---------------------------------------------------------
# Change `object_name` to the stored procedure you want to inspect.

OBJECT_NAME = "BUILD_END_TABLE_TRICKY_FIXED_v3()"   # <- edit this

final_state = fetch_stored_procedure_etl(object_name=OBJECT_NAME)

print("\n" + "=" * 60)
print(f"Status  : {final_state['status']}")
print(f"Message : {final_state['message']}")
if final_state["status"] == "done":
    print("\n-- Extracted ETL script ----------------------------------")
    print(final_state["etl_code"])

print("\nStructured result:")
print(json.dumps(final_state, indent=2, ensure_ascii=False))



Status  : done
Message : ETL script extracted successfully (1883 chars).

-- Extracted ETL script ----------------------------------
def run(session):    from snowflake.snowpark.functions import col    c = session.table("BRONZE.TPCH.CUSTOMER").select("C_CUSTKEY","C_NAME")    o = session.table("BRONZE.TPCH.ORDERS").select("O_ORDERKEY","O_CUSTKEY","O_TOTALPRICE")    l = session.table("BRONZE.TPCH.LINEITEM").select("L_ORDERKEY","L_PARTKEY","L_QUANTITY","L_EXTENDEDPRICE")    j1 = o.join(c, o["O_CUSTKEY"] == c["C_CUSTKEY"], "inner", rsuffix="_c").select(        col("O_ORDERKEY").alias("O_ORDERKEY"),        col("O_CUSTKEY").alias("O_CUSTKEY"),        col("C_CUSTKEY_c").alias("C_CUSTKEY"),        col("C_NAME_c").alias("C_NAME"),        col("O_TOTALPRICE").alias("O_TOTALPRICE")    )    j2 = j1.join(l, j1["O_ORDERKEY"] == l["L_ORDERKEY"], "left", rsuffix="_l").select(        col("O_ORDERKEY").alias("O_ORDERKEY"),        col("O_CUSTKEY").alias("O_CUSTKEY"),        col("C_CUSTKEY").alias("C_CUST